# ResNet과 Transfer Learning

## 학습 목표
- Pretrained ResNet18 이해
- 마지막 분류층 교체
- Transfer Learning
- Fine-tuning
- Confusion Matrix와 F1 평가

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

print(model)

In [ ]:
num_classes = 2
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# 처음에는 backbone을 고정하고 classifier만 학습하는 방법도 사용할 수 있습니다.
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

print(model.fc)

## 실습용 데이터 구조
아래와 같은 폴더 구조의 실제 산업/의료 데이터셋으로 교체할 수 있습니다.

```text
dataset/
  train/
    normal/
    defect/
  val/
    normal/
    defect/
```

In [3]:
transform = weights.transforms()

# 실제 데이터가 준비되어 있다면 아래 경로를 자신의 데이터 경로로 변경
# train_ds = datasets.ImageFolder("./dataset/train", transform=transform)
# val_ds = datasets.ImageFolder("./dataset/val", transform=transform)
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=32)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

## Fine-tuning
Classifier만 학습한 후 성능이 충분하지 않다면 일부 ResNet Layer의 `requires_grad`를 True로 바꾸어 미세조정할 수 있습니다.

In [4]:
# 예: 마지막 residual block부터 fine-tuning
for name, param in model.named_parameters():
    if name.startswith("layer4") or name.startswith("fc"):
        param.requires_grad = True

trainable = [name for name, p in model.named_parameters() if p.requires_grad]
print("Trainable parameters:")
for name in trainable:
    print(name)

Trainable parameters:
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias


## 실무 과제
- Pretrained 모델을 사용하는 이유를 설명
- Classifier-only training과 fine-tuning의 차이 비교
- 클래스 불균형이 있다면 Accuracy 외에 Precision/Recall/F1을 함께 확인